In [1]:
from system_check import *
CUDA_state_print_limited()

CUDA доступна: True
Количество GPU: 4


In [2]:
import torch

def get_device() -> str:
    """
    Возвращает 'cuda' если GPU доступен, иначе 'cpu'.
    """
    return "cuda:1" if torch.cuda.is_available() else "cpu"

In [3]:
device = get_device()
print(f"Используемое устройство: {device}")

Используемое устройство: cuda:1


# **Анализ токенов**

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc

# Очистка памяти от предыдущих моделей
gc.collect()
torch.cuda.empty_cache()

# Загрузка ТОЛЬКО на CPU (без accelerate/device_map)
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token  # критически важно для Qwen

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,      # экономия памяти
    low_cpu_mem_usage=True,         # загрузка без переполнения ОЗУ
    trust_remote_code=False         # не требуется для Qwen2.5
).to(device)  # ← ЯВНОЕ перемещение на CPU

print("✅ Модель загружена. Память:", sum(p.numel() for p in model.parameters()) / 1e9, "B параметров")

/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.60it/s]


✅ Модель загружена. Память: 3.085938688 B параметров


In [12]:
def answer_question_qwen(
    context: str, 
    question: str, 
    max_new_tokens: int = 150,
    max_context_tokens: int = 16384  # ← НАСТРОЙКА: лимит контекста
) -> str:
    """
    Получает ответ от Qwen2.5-3B с поддержкой длинного контекста.
    
    Args:
        context: Текст для анализа (до 128K токенов)
        question: Вопрос к тексту
        max_new_tokens: Макс. длина генерируемого ответа
        max_context_tokens: Лимит токенов контекста (рекомендуется 8192–16384 для ваших данных)
    """
    # === 1. Обрезка контекста ДО формирования промпта ===
    tokens = tokenizer.encode(context, add_special_tokens=False)
    if len(tokens) > max_context_tokens:
        # Сохраняем КОНЕЦ текста (часто содержит выводы/ответы)
        tokens = tokens[-max_context_tokens:]
        context = tokenizer.decode(tokens, skip_special_tokens=True)
    
    # === 2. Формирование промпта ===
    messages = [{
        "role": "user",
        "content": (
            "Ответь строго по тексту ниже. Если информации нет — напиши 'Не указано в тексте'.\n\n"
            f"Текст:\n{context}\n\n"
            f"Вопрос: {question}"
        )
    }]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # === 3. Токенизация с расширенным лимитом ===
    # Важно: max_length должен быть >= max_context_tokens + длина вопроса + системные токены
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_context_tokens + 512  # +512 на вопрос и системные токены
    ).to(device)
    
    # === 4. Генерация ===
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )
    
    # === 5. Извлечение ответа ===
    response_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=True)
    
    return response.strip()

In [4]:
import pandas as pd
def load_dataframe_from_json(file_path: str, 
                   orient: str = 'records') -> pd.DataFrame:
    """
    Простая загрузка JSON в DataFrame без дополнительных проверок
    
    Параметры:
        file_path: путь к JSON файлу
        orient: формат JSON ('records', 'split', 'index', 'columns', 'values')
    
    Возвращает:
        Загруженный DataFrame
    
    Исключения:
        FileNotFoundError: если файл не существует
        ValueError: если JSON невалидный
    """
    try:
        return pd.read_json(file_path, orient=orient, lines=True)
    except FileNotFoundError:
        raise FileNotFoundError(f"File {file_path} not found")
    except ValueError as e:
        try: 
            return pd.read_json(file_path)
        except Exception as e:
            raise ValueError(f"Error while loading JSON: {str(e)}")

In [5]:
datasets_results_path = "../datasets_results"
res_name = "results_2.json"

df_res = load_dataframe_from_json(f"{datasets_results_path}/{res_name}")

df_res.head()

,doc_id,original_text,tasks,tasks_cleaned,first_answer,corrected_answer,keywords,metrics_orig,metrics_new
0,000f90380d768a85e2316225854fc377c079b5c4,Full - Resolution Residual Networks for Semant...,"[Semantic_Segmentation, Real-Time_Semantic_Seg...","[Semantic Segmentation, Real-Time Semantic Seg...",To improve segmentation performance at object ...,Semantic image segmentation,"['score', 'accuracy', 'pretraine', 'geometry']","{'cosine_sim': 0.6268187761306763, 'lev_dist':...","{'cosine_sim': 0.9270111322402954, 'lev_dist': 7}"
1,0012de6bec1f25599e4f02517637e531a71909b9,document: V - Net: Fully Convolutional Neural ...,[Volumetric_Medical_Image_Segmentation],[Volumetric Medical Image Segmentation],"classification, segmentation and object detect...",--None--,--None--,"{'cosine_sim': 0.5885287523269653, 'lev_dist':...","{'cosine_sim': 0.5885287523269653, 'lev_dist':..."
2,007ab5528b3bd310a80d553cccad4b78dc496b02,document: Bi - Directional Attention Flow for ...,"[Question_Answering, Open-Domain_Question_Answ...","[Question Answering, Open-Domain Question Answ...",answering the query.,question answering,"['enable', 'improvement', 'year', 'allow', 'na...","{'cosine_sim': 0.8998040556907654, 'lev_dist':...","{'cosine_sim': 0.9999996423721313, 'lev_dist': 2}"
3,0095c269e7d0c990249312687fc43521019809c4,document: Modelling Interaction of Sentence Pa...,[Natural_Language_Inference],[Natural Language Inference],determine the semantic relationship between tw...,--None--,--None--,"{'cosine_sim': 0.6914587616920471, 'lev_dist':...","{'cosine_sim': 0.6914587616920471, 'lev_dist':..."
4,00b1cdc5bd77bf27f9b1ca630365eeeb456913b4,document: Mastering Chess and Shogi by Self - ...,"[Game_of_Go, Game_of_Shogi]","[Game of Go, Game of Shogi]",search is focused on promising variations,--None--,--None--,"{'cosine_sim': 0.5607925653457642, 'lev_dist':...","{'cosine_sim': 0.5607925653457642, 'lev_dist':..."


In [15]:
import pandas as pd
import time
from tqdm import tqdm

# Создаём колонку для ответов, если её нет
if 'qwen2.5-3B_answer' not in df_res.columns:
    df_res['qwen2.5-3B_answer'] = pd.NA

# Вопрос для всех текстов
question = "Which task was solved? Give a short answer without explanation"

# Прогресс-бар
pbar = tqdm(
    total=len(df_res),
    desc="Обработка текстов Qwen2.5-3B",
    unit="текст",
    ncols=80
)

# Счётчики
success_count = 0
error_count = 0

try:
    for idx in range(len(df_res)):
        # Пропускаем уже обработанные строки
        if pd.notna(df_res.loc[idx, 'qwen2.5-3B_answer']) and df_res.loc[idx, 'qwen2.5-3B_answer'] != '':
            success_count += 1
            pbar.update(1)
            continue
        
        context = df_res.loc[idx, 'original_text']
        
        try:
            # Получаем ответ через вашу функцию
            start_time = time.time()
            answer = answer_question_qwen(
                context=context,
                question=question,
                max_new_tokens=100,        # короткий ответ как в примере
                max_context_tokens=16384   # полный контекст для ваших длинных текстов
            )
            elapsed = time.time() - start_time
            
            # Сохраняем ответ
            df_res.loc[idx, 'qwen2.5-3B_answer'] = answer
            success_count += 1
            
            # Обновляем прогресс-бар
            pbar.set_postfix({
                'успех': success_count,
                'ошибок': error_count,
                'время': f"{elapsed:.1f}с"
            })
            
        except Exception as e:
            error_count += 1
            error_msg = f"[ERROR] {str(e)[:60]}"
            df_res.loc[idx, 'qwen2.5-3B_answer'] = error_msg
            pbar.set_postfix({'ошибок': error_count})
            print(f"\n⚠️ Строка {idx}: {error_msg}")
        
        pbar.update(1)
        
        # Пауза для стабильности на CPU
        time.sleep(0.3)
        
        # Промежуточное сохранение каждые 10 строк
        if (idx + 1) % 10 == 0:
            df_res.to_csv('df_res_with_qwen_answers.csv', index=False)
            pbar.write(f" 💾 Сохранено {idx + 1}/{len(df_res)} строк")

except KeyboardInterrupt:
    pbar.close()
    print("\n\n🛑 Обработка прервана. Сохраняем результаты...")
    df_res.to_csv('df_res_with_qwen_answers.csv', index=False)
    print("✅ Промежуточные результаты сохранены в 'df_res_with_qwen_answers.csv'")
    raise

finally:
    pbar.close()

# Финальное сохранение
df_res.to_csv('df_res_with_qwen_answers.csv', index=False)

# Итоговая статистика
print("\n" + "="*70)
print("✅ ОБРАБОТКА ЗАВЕРШЕНА")
print(f"   Всего строк:    {len(df_res)}")
print(f"   Успешно:        {success_count} ({success_count/len(df_res)*100:.1f}%)")
print(f"   Ошибок:         {error_count} ({error_count/len(df_res)*100:.1f}%)")
print(f"   Результаты:     df_res_with_qwen_answers.csv")
print("="*70)

# Показать примеры ответов
print("\n🔍 Примеры ответов:")
for i in range(min(5, len(df_res))):
    print(f"\nСтрока {i}:")
    print(f"   Ответ: {df_res.loc[i, 'qwen2.5-3B_answer'][:100]}...")

Обработка текстов Qwen2.5-3B:   3%| | 10/306 [00:51<34:36,  7.02s/текст, успех=1The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 💾 Сохранено 10/306 строк


Обработка текстов Qwen2.5-3B:   7%| | 20/306 [01:44<23:15,  4.88s/текст, успех=2The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 💾 Сохранено 20/306 строк


Обработка текстов Qwen2.5-3B:  10%| | 30/306 [02:24<23:43,  5.16s/текст, успех=3The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

 💾 Сохранено 290/306 строк


Обработка текстов Qwen2.5-3B:  98%|▉| 300/306 [23:59<00:25,  4.29s/текст, успех=The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 💾 Сохранено 300/306 строк


Обработка текстов Qwen2.5-3B: 100%|▉| 305/306 [24:17<00:03,  3.86s/текст, успех=The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Обработка текстов Qwen2.5-3B: 100%|█| 306/306 [24:21<00:00,  4.78s/текст, успех=


✅ ОБРАБОТКА ЗАВЕРШЕНА
   Всего строк:    306
   Успешно:        306 (100.0%)
   Ошибок:         0 (0.0%)
   Результаты:     df_res_with_qwen_answers.csv

🔍 Примеры ответов:

Строка 0:
   Ответ: Semantic image segmentation...

Строка 1:
   Ответ: Volumetric medical image segmentation...

Строка 2:
   Ответ: Question Answering...

Строка 3:
   Ответ: Matching question and answer...

Строка 4:
   Ответ: Mastering chess and shogi...


In [16]:
df_res.head()

,doc_id,original_text,tasks,tasks_cleaned,first_answer,corrected_answer,keywords,metrics_orig,metrics_new,qwen2.5-3B_answer
0,000f90380d768a85e2316225854fc377c079b5c4,Full - Resolution Residual Networks for Semant...,"[Semantic_Segmentation, Real-Time_Semantic_Seg...","[Semantic Segmentation, Real-Time Semantic Seg...",To improve segmentation performance at object ...,Semantic image segmentation,"['score', 'accuracy', 'pretraine', 'geometry']","{'cosine_sim': 0.6268187761306763, 'lev_dist':...","{'cosine_sim': 0.9270111322402954, 'lev_dist': 7}",Semantic image segmentation
1,0012de6bec1f25599e4f02517637e531a71909b9,document: V - Net: Fully Convolutional Neural ...,[Volumetric_Medical_Image_Segmentation],[Volumetric Medical Image Segmentation],"classification, segmentation and object detect...",--None--,--None--,"{'cosine_sim': 0.5885287523269653, 'lev_dist':...","{'cosine_sim': 0.5885287523269653, 'lev_dist':...",Volumetric medical image segmentation
2,007ab5528b3bd310a80d553cccad4b78dc496b02,document: Bi - Directional Attention Flow for ...,"[Question_Answering, Open-Domain_Question_Answ...","[Question Answering, Open-Domain Question Answ...",answering the query.,question answering,"['enable', 'improvement', 'year', 'allow', 'na...","{'cosine_sim': 0.8998040556907654, 'lev_dist':...","{'cosine_sim': 0.9999996423721313, 'lev_dist': 2}",Question Answering
3,0095c269e7d0c990249312687fc43521019809c4,document: Modelling Interaction of Sentence Pa...,[Natural_Language_Inference],[Natural Language Inference],determine the semantic relationship between tw...,--None--,--None--,"{'cosine_sim': 0.6914587616920471, 'lev_dist':...","{'cosine_sim': 0.6914587616920471, 'lev_dist':...",Matching question and answer
4,00b1cdc5bd77bf27f9b1ca630365eeeb456913b4,document: Mastering Chess and Shogi by Self - ...,"[Game_of_Go, Game_of_Shogi]","[Game of Go, Game of Shogi]",search is focused on promising variations,--None--,--None--,"{'cosine_sim': 0.5607925653457642, 'lev_dist':...","{'cosine_sim': 0.5607925653457642, 'lev_dist':...",Mastering chess and shogi


In [17]:
df_res11 = pd.read_csv('df_res_with_qwen_answers.csv')
df_res11.head()

,doc_id,original_text,tasks,tasks_cleaned,first_answer,corrected_answer,keywords,metrics_orig,metrics_new,qwen2.5-3B_answer
0,000f90380d768a85e2316225854fc377c079b5c4,Full - Resolution Residual Networks for Semant...,"['Semantic_Segmentation', 'Real-Time_Semantic_...","['Semantic Segmentation', 'Real-Time Semantic ...",To improve segmentation performance at object ...,Semantic image segmentation,"['score', 'accuracy', 'pretraine', 'geometry']","{'cosine_sim': 0.6268187761306763, 'lev_dist':...","{'cosine_sim': 0.9270111322402954, 'lev_dist': 7}",Semantic image segmentation
1,0012de6bec1f25599e4f02517637e531a71909b9,document: V - Net: Fully Convolutional Neural ...,['Volumetric_Medical_Image_Segmentation'],['Volumetric Medical Image Segmentation'],"classification, segmentation and object detect...",--None--,--None--,"{'cosine_sim': 0.5885287523269653, 'lev_dist':...","{'cosine_sim': 0.5885287523269653, 'lev_dist':...",Volumetric medical image segmentation
2,007ab5528b3bd310a80d553cccad4b78dc496b02,document: Bi - Directional Attention Flow for ...,"['Question_Answering', 'Open-Domain_Question_A...","['Question Answering', 'Open-Domain Question A...",answering the query.,question answering,"['enable', 'improvement', 'year', 'allow', 'na...","{'cosine_sim': 0.8998040556907654, 'lev_dist':...","{'cosine_sim': 0.9999996423721313, 'lev_dist': 2}",Question Answering
3,0095c269e7d0c990249312687fc43521019809c4,document: Modelling Interaction of Sentence Pa...,['Natural_Language_Inference'],['Natural Language Inference'],determine the semantic relationship between tw...,--None--,--None--,"{'cosine_sim': 0.6914587616920471, 'lev_dist':...","{'cosine_sim': 0.6914587616920471, 'lev_dist':...",Matching question and answer
4,00b1cdc5bd77bf27f9b1ca630365eeeb456913b4,document: Mastering Chess and Shogi by Self - ...,"['Game_of_Go', 'Game_of_Shogi']","['Game of Go', 'Game of Shogi']",search is focused on promising variations,--None--,--None--,"{'cosine_sim': 0.5607925653457642, 'lev_dist':...","{'cosine_sim': 0.5607925653457642, 'lev_dist':...",Mastering chess and shogi


In [14]:
# Пример применения к вашему датафрейму
context = df_res.iloc[0]['original_text']
question = "Which task was solved? Give a short answer without explanation"

print("📄 Длина текста:", len(tokenizer.encode(context, add_special_tokens=False)), "токенов")
print("\n❓ Вопрос:", question)
print("\n✅ Ответ:")
print(answer_question_qwen(context, question))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


📄 Длина текста: 7067 токенов

❓ Вопрос: Which task was solved? Give a short answer without explanation

✅ Ответ:
Semantic image segmentation


In [6]:
from transformers import *
roberta_qa_dir = "/home/pmartynyuk/UnTIE project/scripts/models_processing/models/bert_eng_qa_baseroberta_model"

model = AutoModelForQuestionAnswering.from_pretrained(roberta_qa_dir, output_attentions=True).to(device)
concrete_tokenizer = AutoTokenizer.from_pretrained(roberta_qa_dir)

/home/pmartynyuk/UnTIE project/untie_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
loading configuration file /home/pmartynyuk/UnTIE project/scripts/models_processing/models/bert_eng_qa_baseroberta_model/config.json
Model config RobertaConfig {
  "architectures": [
    "RobertaForQuestionAnswering"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "language": "english",
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "name": "Roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_attentions": true,
  "

In [7]:
from transformers import pipeline
import torch

# Загрузка модели (весит всего ~50 МБ)
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=concrete_tokenizer,
    device=1 if torch.cuda.is_available() else -1,  # -1 = CPU
    max_seq_len=512,      # максимальная длина чанка
    doc_stride=128,       # перекрытие между чанками (рекомендуется 128)
    max_answer_len=100    # макс. длина извлекаемого ответа
)

Device set to use cuda:1


In [8]:
idx = 0

context = str(df_res.loc[idx, 'original_text'])
question = "Which task was solved?"

In [9]:
result = qa_pipeline(
            question=question,
            context=context,
            handle_impossible_answer=True,
            max_seq_len=512,
            doc_stride=128
        )
result

Disabling tokenizer parallelism, we're using DataLoader multithreading already
RobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


{'score': 0.33691877126693726, 'start': 0, 'end': 0, 'answer': ''}

In [46]:
def answer_question_roberta(context: str, question: str) -> str:
    """
    Надёжная функция для RoBERTa QA. Всегда возвращает непустую строку.
    """
    # === 1. Валидация входных данных ===
    if not isinstance(context, str) or not context.strip():
        return "ERROR: пустой или некорректный текст"
    if not isinstance(question, str) or not question.strip():
        return "ERROR: пустой вопрос"
    
    # === 2. Вызов модели ===
    try:
        result = qa_pipeline(
            question=question,
            context=context,
            handle_impossible_answer=True,
            max_seq_len=512,
            doc_stride=128
        )
    except Exception as e:
        return f"ERROR: pipeline exception ({type(e).__name__})"
    
    # === 3. Обработка результата ===
    try:
        # Список ответов (длинный текст)
        if isinstance(result, list) and result:
            # Фильтруем только словари с ответами
            valid_answers = [
                r for r in result 
                if isinstance(r, dict) 
                and 'answer' in r 
                and isinstance(r['answer'], str)
                # and len(r['answer'].strip()) >= 2  # минимум 2 символа
                # and r.get('score', 0) > 0.001       # любой ненулевой скор
            ]
            
            if valid_answers:
                # Берём ответ с максимальным скором
                best = max(valid_answers, key=lambda x: x.get('score', 0))
                answer = best['answer'].strip()
                # Ограничиваем длину ответа
                return answer[:150] if answer else "Ответ не найден"
            else:
                return "Ответ не найден"
        
        # Один ответ (короткий текст)
        elif isinstance(result, dict) and 'answer' in result:
            answer = str(result.get('answer', '')).strip()
            if len(answer) >= 2 and result.get('score', 0) > 0.001:
                return answer[:150]
            else:
                return "Ответ не найден"
        
        # Любой другой случай
        else:
            return "Ответ не найден"
    
    except Exception as e:
        return f"ERROR: processing exception ({type(e).__name__})"

In [47]:
df_res = df_res.drop(columns=['roberta_answer'])

In [48]:
import pandas as pd
import time
from tqdm import tqdm

# === КРИТИЧЕСКИ ВАЖНО: Очистка колонки от существующих NaN ===
if 'roberta_answer' in df_res.columns:
    df_res['roberta_answer'] = df_res['roberta_answer'].fillna("").astype(str)
else:
    df_res['roberta_answer'] = ""

# Убедимся, что original_text не содержит NaN
df_res['original_text'] = df_res['original_text'].fillna("")

question = "Which task was solved?"

pbar = tqdm(total=len(df_res), desc="RoBERTa QA", unit="текст", ncols=80)

for idx in range(len(df_res)):
    # Пропускаем уже обработанные (не "Ответ не найден" и не ошибки)
    current = str(df_res.loc[idx, 'roberta_answer']).strip()
    if current and not current.startswith(("Ответ не найден", "ERROR")):
        pbar.update(1)
        continue
    
    context = str(df_res.loc[idx, 'original_text'])
    
    # Получаем ответ (гарантированно строка)
    answer = answer_question_roberta(context, question)
    
    # Принудительно сохраняем как строку
    df_res.loc[idx, 'roberta_answer'] = str(answer)
    
    pbar.update(1)
    
    # Промежуточное сохранение
    if (idx + 1) % 10 == 0:
        df_res.to_csv('df_res_with_roberta.csv', index=False)
        pbar.set_postfix({"последний": answer[:30]})

pbar.close()
df_res.to_csv('df_res_with_roberta.csv', index=False)

# Финальная статистика
total = len(df_res)
not_found = df_res['roberta_answer'].str.contains("Ответ не найден", na=False).sum()
errors = df_res['roberta_answer'].str.startswith("ERROR", na=False).sum()
valid = total - not_found - errors
nan_check = df_res['roberta_answer'].isna().sum()

print("\n" + "="*70)
print("✅ ОБРАБОТКА ЗАВЕРШЕНА")
print(f"   Всего строк:          {total}")
print(f"   Валидных ответов:     {valid} ({valid/total*100:.1f}%)")
print(f"   'Ответ не найден':    {not_found} ({not_found/total*100:.1f}%)")
print(f"   Ошибок:               {errors} ({errors/total*100:.1f}%)")
print(f"   NaN в колонке:        {nan_check} ← ДОЛЖНО БЫТЬ 0")
print("="*70)

# Примеры хороших ответов
print("\n🔍 Примеры найденных ответов:")
valid_answers = df_res[~df_res['roberta_answer'].str.contains("Ответ не найден|ERROR", na=False)]
for i, (_, row) in enumerate(valid_answers.head(5).iterrows()):
    print(f"{i+1}. {row['roberta_answer']}")

RoBERTa QA: 100%|█| 306/306 [01:30<00:00,  3.37текст/s, последний=Ответ не найде


✅ ОБРАБОТКА ЗАВЕРШЕНА
   Всего строк:          306
   Валидных ответов:     63 (20.6%)
   'Ответ не найден':    243 (79.4%)
   Ошибок:               0 (0.0%)
   NaN в колонке:        0 ← ДОЛЖНО БЫТЬ 0

🔍 Примеры найденных ответов:
1. classification, segmentation and object detection
2. 1 to 7
3. 3D facial landmark localisation
4. estimating image distribution
5. Dense Human Pose Estimation


In [ ]:
from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

In [ ]:
import pandas as pd

# Пример датафрейма (замените на ваш)
# df = pd.read_csv('your_dataset.csv')

# # Функция подсчёта токенов БЕЗ специальных токенов (<BOS>/<EOS>)
# def count_tokens(text: str) -> int:
#     if pd.isna(text) or not isinstance(text, str):
#         return 0
#     # return_tensors=None → возвращаем список, не тензор
#     tokens = tokenizer.encode(text, add_special_tokens=False)
#     return len(tokens)

# # Применяем к колонке
# df_res['token_count'] = df_res['original_text'].apply(count_tokens)

In [ ]:
df_res.head()

In [ ]:
# # Базовая статистика
# print("Статистика длин текстов в токенах:")
# print(df_res['token_count'].describe())

# # Критические пороги для модели 0.5B
# print("\n⚠️ Рекомендации для Qwen2.5-0.5B:")
# print(f"  • Текстов > 256 токенов: {(df_res['token_count'] > 256).sum()} ({(df_res['token_count'] > 256).mean()*100:.1f}%)")
# print(f"  • Текстов > 512 токенов: {(df_res['token_count'] > 512).sum()} ({(df_res['token_count'] > 512).mean()*100:.1f}%)")
# print(f"  • Максимальная длина: {df_res['token_count'].max()} токенов")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "microsoft/Phi-3-mini-128k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="eager",  # ← КЛЮЧЕВОЙ ПАРАМЕТР для обхода ошибки
).to(device)

In [ ]:
def answer_question_phi3(context: str, question: str, max_new_tokens: int = 200) -> str:
    """
    Получает ответ от Phi-3-mini-128k-instruct на вопрос по тексту.
    Работает без ошибки DynamicCache благодаря use_cache=False.
    """
    # Упрощённый промпт без применения шаблона (избегаем проблем с apply_chat_template)
    prompt = f"""<|user|>
Текст: {context}

Вопрос: {question}<|end|>
<|assistant|>"""
    
    # Токенизация с безопасным лимитом для CPU
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=8192  # достаточно для ваших текстов (макс. 15,752 токенов)
    ).to(device)
    
    # Генерация с ОБХОДОМ ошибки DynamicCache
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=False  # ← КЛЮЧЕВОЙ ПАРАМЕТР для обхода ошибки
        )
    
    # Декодирование и очистка ответа
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Извлекаем только часть после <|assistant|>
    if "<|assistant|>" in full_text:
        answer = full_text.split("<|assistant|>")[-1].strip()
    else:
        # Fallback: убираем промпт
        prompt_clean = tokenizer.decode(inputs.input_ids[0], skip_special_tokens=True)
        answer = full_text.replace(prompt_clean, "").strip()
    
    # Убираем возможные повторы промпта в ответе
    if "Вопрос:" in answer:
        answer = answer.split("Вопрос:")[0].strip()
    
    return answer.strip()

In [ ]:
# Пример применения к вашему датафрейму
context = df_res.iloc[0]['original_text']
question = "Какова основная тема этого текста?"

print("📄 Длина текста:", len(tokenizer.encode(context, add_special_tokens=False)), "токенов")
print("\n❓ Вопрос:", question)
print("\n✅ Ответ:")
print(answer_question_phi3(context, question))

In [ ]:
import torch, psutil, os

print("Устройство модели:", next(model.parameters()).device)
print("Тип данных модели:", next(model.parameters()).dtype)
print(f"Свободная ОЗУ: {psutil.virtual_memory().available / 1024**3:.1f} ГБ из {psutil.virtual_memory().total / 1024**3:.1f} ГБ")
print(f"Используется swap: {psutil.swap_memory().used / 1024**3:.1f} ГБ")